In [44]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
import joblib
import pandas as pd
from datetime import datetime

In [45]:
data = pd.read_csv("../dataset/dataset.csv", dtype={'teamId' : str})
data_elo = pd.read_csv("../dataset/nba_elo.csv", dtype={'teamId' : str})
data


In [46]:
nb_games="10"
df_last_10_games = data[data['teamId'] == "1610612748"].tail(10)
df_last_10_games
df_stat_last_perf_home = pd.DataFrame(columns=['NB_WIN_L10', 'PCT_TIR_REUSSI_L10','PCT_3PTS_L10', 
                           'PCT_LANCER_FRANC_L10','estimatedOffensiveRating_L10', 'offensiveRating_L10',
                           'estimatedDefensiveRating_L10',
                           'defensiveRating_L10','estimatedNetRating_L10', 'netRating_L10', 'assistPercentage_L10',
                           'assistToTurnover_L10', 'assistRatio_L10','estimatedTeamTurnoverPercentage_L10', 'turnoverRatio_L10',
                           'effectiveFieldGoalPercentage_L10','trueShootingPercentage_L10', 'estimatedPace_L10', 'pace_L10',
                           'pacePer40_L10', 'PIE_L10'])
df_last_10_games.reset_index(drop=True, inplace=True)
new_row = {"NB_WIN_L"+nb_games : (df_last_10_games['WL'] == 'W').sum() / int(nb_games),
                   "PCT_TIR_REUSSI_L"+nb_games : df_last_10_games['PCT_TIR_REUSSI'].mean(),
                   "PCT_3PTS_L"+nb_games : df_last_10_games['PCT_3PTS'].mean(),
                   "PCT_LANCER_FRANC_L"+nb_games : df_last_10_games['PCT_LANCER_FRANC'].mean(),
                   "estimatedOffensiveRating_L"+nb_games : df_last_10_games['estimatedOffensiveRating'].mean(),
                   "offensiveRating_L"+nb_games : df_last_10_games['offensiveRating'].mean(),
                   "estimatedDefensiveRating_L"+nb_games : df_last_10_games['estimatedDefensiveRating'].mean(),
                   "defensiveRating_L"+nb_games : df_last_10_games['defensiveRating'].mean(),
                   "estimatedNetRating_L"+nb_games : df_last_10_games['estimatedNetRating'].mean(),
                   "netRating_L"+nb_games : df_last_10_games['netRating'].mean(),
                   "assistPercentage_L"+nb_games : df_last_10_games['assistPercentage'].mean(),
                   "assistToTurnover_L"+nb_games : df_last_10_games['assistToTurnover'].mean(),
                   "assistRatio_L"+nb_games : df_last_10_games['assistRatio'].mean(),
                   "estimatedTeamTurnoverPercentage_L"+nb_games : df_last_10_games['estimatedTeamTurnoverPercentage'].mean(),
                   "turnoverRatio_L"+nb_games : df_last_10_games['turnoverRatio'].mean(),
                   "effectiveFieldGoalPercentage_L"+nb_games : df_last_10_games['effectiveFieldGoalPercentage'].mean(),
                   "trueShootingPercentage_L"+nb_games : df_last_10_games['trueShootingPercentage'].mean(),
                   "estimatedPace_L"+nb_games : df_last_10_games['estimatedPace'].mean(),
                   "pace_L"+nb_games : df_last_10_games['pace'].mean(),
                   "pacePer40_L"+nb_games : df_last_10_games['pacePer40'].mean(),
                   "PIE_L"+nb_games : df_last_10_games['PIE'].mean()}
df_stat_last_perf_home.loc[len(df_stat_last_perf_home)] = new_row
df_stat_last_perf_home

In [48]:
def get_elo(team_name, data_elo):
    df_elo_home = data_elo[data_elo['team1'] == "ATL"]
    df_elo_away = data_elo[data_elo['team2'] == "ATL"]
    
    if datetime.strptime(df_elo_home['date'].iloc[-1], '%Y-%m-%d') > datetime.strptime(df_elo_away['date'].iloc[-1], '%Y-%m-%d'):
        return df_elo_home['elo1_post'].iloc[-1]
    else :
        return df_elo_away['elo2_post'].iloc[-1]

df_stat_last_perf_home['elo'] = get_elo('ATL', data_elo)
df_stat_last_perf_home

In [49]:

df_stat_last_perf['a'] = [1]
df_stat_last_perf

In [38]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import make_scorer, accuracy_score, f1_score, confusion_matrix
import joblib

data = pd.read_csv('../dataset/final_dataset.csv', parse_dates=['GAME_DATE'], dtype={'gameId' : str, 'H_teamId' : str, 'A_teamId' : str,})
data = data.round(2)

condition = (data['GAME_DATE'] > pd.to_datetime('2023-09-01')) & (data['GAME_DATE'] < pd.to_datetime('2024-09-01'))
data_test = data[condition]
data_train = data[~condition]

X_train = data_train.drop(columns=['HOME_WON', 'GAME_DATE','gameId', 'A_teamId', 'H_teamId','H_teamId','A_teamId','H_POINTS', 'A_POINTS','H_teamName', 'A_teamName','trueShootingPercentage_L10', 'PCT_TIR_REUSSI_L10', 'effectiveFieldGoalPercentage_L10','NB_WIN_L10', 'PCT_3PT_L10', 'W_CONFR_D'])  # Fonctionnalités
y_train = data_train['HOME_WON']  # Cible

X_test = data_test.drop(columns=['HOME_WON', 'GAME_DATE','gameId', 'A_teamId', 'H_teamId','H_teamId','A_teamId','H_POINTS', 'A_POINTS','H_teamName', 'A_teamName','trueShootingPercentage_L10', 'PCT_TIR_REUSSI_L10', 'effectiveFieldGoalPercentage_L10', 'NB_WIN_L10', 'PCT_3PT_L10', 'W_CONFR_D'])  # Fonctionnalités
y_test = data_test['HOME_WON']

In [13]:
X_train

In [14]:
import pandas as pd

# Charger le dataset
nba_elo_df = pd.read_csv('../dataset/nba_elo.csv')

# Calculer les probabilités ELO pour chaque match
nba_elo_df['calculated_elo_prob1'] = 1 / (1 + 10 ** ((nba_elo_df['elo2_pre'] - nba_elo_df['elo1_pre']) / 400))
nba_elo_df['calculated_elo_prob2'] = 1 / (1 + 10 ** ((nba_elo_df['elo1_pre'] - nba_elo_df['elo2_pre']) / 400))

# Comparer les résultats calculés avec ceux du dataset
nba_elo_df[['elo_prob1', 'calculated_elo_prob1', 'elo_prob2', 'calculated_elo_prob2']].head(30)
